In [1]:
import quadrants as qd

qd.init(arch=qd.amdgpu)


[Quadrants] version 1.3.0, llvm 22.1.0, commit ab9a58ab, linux, python 3.12.3


[I 09/23/26 05:29:37.551 3340212] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


[Quadrants] Starting on arch=amdgpu


In [ ]:
import quadrants as qd

@qd.kernel
def hello(a: qd.types.NDArray[qd.i32, 1]) -> None:
    for i in range(10):
        a[i] = i * 2

a = qd.ndarray(qd.i32, (10, ))
hello(a)
print(a)

<qd.ndarray>


In [3]:
dir(qd)

['Backend',
 'BitpackedFields',
 'BufferView',
 'CRITICAL',
 'DEBUG',
 'DeviceCapability',
 'ERROR',
 'Event',
 'Field',
 'FieldsBuilder',
 'Format',
 'GraphStatus',
 'INFO',
 'Layout',
 'Matrix',
 'MatrixField',
 'MatrixNdarray',
 'MatrixTensor',
 'Mesh',
 'MeshInstance',
 'Ndarray',
 'QuadrantsAssertionError',
 'QuadrantsCompilationError',
 'QuadrantsNameError',
 'QuadrantsRuntimeError',
 'QuadrantsRuntimeTypeError',
 'QuadrantsSyntaxError',
 'QuadrantsTypeError',
 'SNode',
 'ScalarField',
 'ScalarNdarray',
 'Stream',
 'Struct',
 'StructField',
 'TRACE',
 'Template',
 'Tensor',
 'Vector',
 'VectorNdarray',
 'VectorTensor',
 'WARN',
 '_MatrixTensor',
 '_TYPE_CHECKING',
 '_Tensor',
 '_VectorTensor',
 '__all__',
 '__annotations__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__getattr__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 '__version_str__',
 '_funcs',
 '_kernels',
 '_lib',
 '_logging',
 '_python_primitive_types',
 '_qd_

In [ ]:
# First quadrants kernel
import numpy as np
import numpy.typing as npt
import time

def lcg_np(B: int, lcg_its: int, a: npt.NDArray) -> None:
    for i in range(B):
        x = a[i]
        for j in range(lcg_its):
            x = (1664525 * x + 1013904223) % 2147483647
        a[i] = x

B = 16000
a = np.ndarray((B, ), np.int32)

start = time.time()
lcg_np(B, 1000, a)
end = time.time()
print(f"Time taken: {end - start} seconds")

# /tmp/ipykernel_3340212/1173608805.py:10: RuntimeWarning: overflow encountered in scalar multiply
#   x = (1664525 * x + 1013904223) % 2147483647
# /tmp/ipykernel_3340212/1173608805.py:10: RuntimeWarning: overflow encountered in scalar add
#   x = (1664525 * x + 1013904223) % 2147483647
# Time taken: 7.880692481994629 seconds

/tmp/ipykernel_3340212/1173608805.py:10: RuntimeWarning: overflow encountered in scalar multiply
  x = (1664525 * x + 1013904223) % 2147483647
/tmp/ipykernel_3340212/1173608805.py:10: RuntimeWarning: overflow encountered in scalar add
  x = (1664525 * x + 1013904223) % 2147483647


Time taken: 7.880692481994629 seconds


In [ ]:
@qd.kernel
def lcg_ti(B: int, lcg_its: int, a: qd.types.NDArray[qd.i32, 1]) -> None:
    for i in range(B):
        x = a[i]
        for j in range(lcg_its):
            x = (1664525 * x + 1013904223) % 2147483647
        a[i] = x

qd.init(arch=qd.amdgpu)

B = 16000
a = qd.ndarray(qd.i32, (B, ))

qd.sync()
start = time.time()

lcg_ti(B, 1000, a)

qd.sync()
end = time.time()
print(f"Time taken: {end - start} seconds")

# [Quadrants] Starting on arch=amdgpu
# Time taken: 0.22856926918029785 seconds
# 7.88 sec vs 0.23 sec => 34x faster
#
# Compared to NVidia box, 5090,  6.9 sec vs. 0.02 sec => 345x faster

[Quadrants] Starting on arch=amdgpu
Time taken: 0.22856926918029785 seconds


In [16]:
7.88 / 0.22857

34.475215470096686

## Tutorials Examples
### Hello Genesis

In [1]:
import subprocess
from pathlib import Path

import numpy as np
from PIL import Image

import genesis as gs

gs.init(backend=gs.amdgpu)

scene = gs.Scene(show_viewer=False)

plane = scene.add_entity(
    gs.morphs.Plane(),
)
franka = scene.add_entity(
    gs.morphs.MJCF(file="xml/franka_emika_panda/panda.xml"),
)

# Generic three-quarter viewpoint aimed at the robot base.
cam = scene.add_camera(res=(640, 480), pos=(0, 1.5, 2.5), lookat=(0.0, 0.0, 0.4), fov=40)    # pos=(2.0, 1.5, 1.5)

scene.build()

out_dir = Path("out")
out_dir.mkdir(exist_ok=True)
png_path = out_dir / "hello_genesis_t0.png"
mp4_path = out_dir / "hello_genesis.mp4"

# Scene at t = 0, before any step.
frame = cam.render()[0]
Image.fromarray(frame).save(png_path)

steps, render_every = 1000, 4
height, width = frame.shape[:2]
fps = round(1.0 / (scene.dt * render_every))

# Frames are piped straight into ffmpeg, so no per-step image is written to disk.
ffmpeg = subprocess.Popen(
    [
        "ffmpeg", "-y", "-loglevel", "error",
        "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{width}x{height}", "-framerate", str(fps),
        "-i", "-",
        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18",
        str(mp4_path),
    ],
    stdin=subprocess.PIPE,
)
try:
    for i in range(steps):
        scene.step()
        if i % render_every == 0:
            ffmpeg.stdin.write(np.ascontiguousarray(cam.render()[0]).tobytes())
finally:
    ffmpeg.stdin.close()
    ffmpeg.wait()

print(f"{png_path} | {mp4_path} at {fps} fps, {steps // render_every} frames")


[I 09/24/26 03:04:28.552 3106237] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout
/mnt/dcgpuval/younsroh/projects/3d/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[Genesis] [03:04:31] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [03:04:31] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [03:04:31] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [03:04:32] [INFO] Running on [AMD Instinct MI325X] with backend gs.amdgpu. Device memory: 255.98 GB.
[Genesis] [03:04:32] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.1, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
[Genesis] [03:04:32] [INFO] Scene <02461d2> created.
[Genesis] [03:04:32] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 0, uid: <b013190>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.


/mnt/dcgpuval/younsroh/projects/3d/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[Genesis] [03:04:33] [WARNING] (MJCF) Approximating tendon by joint actuator for `finger_joint1`
[Genesis] [03:04:33] [WARNING] (MJCF) Approximating tendon by joint actuator for `finger_joint2`
[Genesis] [03:04:33] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 1, uid: <b3f1987>, morph: <gs.morphs.MJCF(file='/mnt/dcgpuval/younsroh/projects/3d/.venv/lib/python3.12/site-packages/genesis/assets/xml/franka_emika_panda/panda.xml')>, material: <gs.materials.Rigid>.
[Genesis] [03:04:33] [INFO] Building scene <02461d2>...
[Genesis] [03:04:33] [WARNING] Neutral robot position (qpos0) exceeds joint limits.
[Genesis] [03:04:34] [WARNING] Constraint solver time constant should be greater than 2*substep_dt. timeconst is changed from `0.005` to `0.02`). Decrease simulation timestep or increase timeconst to avoid altering the original value.
[Genesis] [03:04:35] [WARNING] Filtered out geometry pairs causing self-collision for the neutral configuration (qpos0): (6, 11), (7, 11), (14, 21), (15, 2

### Control Your Robot

In [7]:
import os
import subprocess
from pathlib import Path

import numpy as np
from PIL import Image

import genesis as gs

# This node has no hardware GL, so pyrender renders on llvmpipe and Genesis
# disables shadows. llvmpipe can do shadow maps, it just cannot allocate the
# default 8192^2 cube map, so shrink it and re-enable them by hand.
from genesis.ext.pyrender import constants as _pr_const, renderer as _pr_renderer, light as _pr_light

for _m in (_pr_const, _pr_renderer, _pr_light):
    _m.SHADOW_TEX_SZ = 2048

# gs.init(backend=gs.amdgpu)

scene = gs.Scene(
    sim_options=gs.options.SimOptions(
        dt=0.01,
    ),
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(0, -3.5, 2.5),
        camera_lookat=(0.0, 0.0, 0.5),
        camera_fov=30,
    ),
    show_viewer=False,  # True,
    vis_options=gs.options.VisOptions(
        shadow=True,
        ambient_light=(0.15, 0.15, 0.15),
        lights=[
            # keep or drop the default sun
            {"type": "directional", "dir": (-1, -1, -1), "color": (1.0, 1.0, 1.0), "intensity": 8.0},
            # point light above/beside the robot
            {"type": "point", "pos": (1.5, -1.0, 2.0), "color": (1.0, 0.95, 0.85), "intensity": 20.0},
        ],
    ),
)

plane = scene.add_entity(
    gs.morphs.Plane(),
)
franka = scene.add_entity(
    gs.morphs.MJCF(
        file="xml/franka_emika_panda/panda.xml",
    ),
)
# Match the viewer camera so the movie uses the same generic viewpoint.
cam = scene.add_camera(
    res=(640, 480),
    pos=(0, -3.5, 2.5),
    lookat=(0.0, 0.0, 0.5),
    fov=30,
)
scene.build()

# The offscreen renderer drops shadows whenever it sees a software context.
scene.visualizer._rasterizer._renderer._is_software = False

joints_name = (
    "joint1",
    "joint2",
    "joint3",
    "joint4",
    "joint5",
    "joint6",
    "joint7",
    "finger_joint1",
    "finger_joint2",
)
motors_dof_idx = [franka.get_joint(name).dofs_idx_local[0] for name in joints_name]

franka.set_dofs_kp(
    kp=np.array([4500, 4500, 3500, 3500, 2000, 2000, 2000, 100, 100]),
    dofs_idx_local=motors_dof_idx,
)
franka.set_dofs_kv(
    kv=np.array([450, 450, 350, 350, 200, 200, 200, 10, 10]),
    dofs_idx_local=motors_dof_idx,
)
franka.set_dofs_force_range(
    lower=np.array([-87, -87, -87, -87, -12, -12, -12, -100, -100]),
    upper=np.array([87, 87, 87, 87, 12, 12, 12, 100, 100]),
    dofs_idx_local=motors_dof_idx,
)

out_dir = Path("out")
out_dir.mkdir(exist_ok=True)
png_path = out_dir / "control_robot_t0.png"
mp4_path = out_dir / "control_robot.mp4"

frame = cam.render()[0]
Image.fromarray(frame).save(png_path)

render_every = 3
height, width = frame.shape[:2]
fps = round(1.0 / (scene.dt * render_every))
horizon = 1250 if "PYTEST_VERSION" not in os.environ else 5
n_hard_reset = 150

ffmpeg = subprocess.Popen(
    [
        "ffmpeg", "-y", "-loglevel", "error",
        "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{width}x{height}", "-framerate", str(fps),
        "-i", "-",
        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18",
        str(mp4_path),
    ],
    stdin=subprocess.PIPE,
)


def write_frame(step_idx):
    if step_idx % render_every == 0:
        ffmpeg.stdin.write(np.ascontiguousarray(cam.render()[0]).tobytes())


try:
    # Hard reset
    for i in range(n_hard_reset):
        if i < 50:
            franka.set_dofs_position(np.array([1, 1, 0, 0, 0, 0, 0, 0.04, 0.04]), motors_dof_idx)
        elif i < 100:
            franka.set_dofs_position(np.array([-1, 0.8, 1, -2, 1, 0.5, -0.5, 0.04, 0.04]), motors_dof_idx)
        else:
            franka.set_dofs_position(np.array([0, 0, 0, 0, 0, 0, 0, 0, 0]), motors_dof_idx)
        scene.step()
        write_frame(i)

    # PD control
    for i in range(horizon):
        if i == 0:
            franka.control_dofs_position(
                np.array([1, 1, 0, 0, 0, 0, 0, 0.04, 0.04]),
                motors_dof_idx,
            )
        elif i == 250:
            franka.control_dofs_position(
                np.array([-1, 0.8, 1, -2, 1, 0.5, -0.5, 0.04, 0.04]),
                motors_dof_idx,
            )
        elif i == 500:
            franka.control_dofs_position(
                np.array([0, 0, 0, 0, 0, 0, 0, 0, 0]),
                motors_dof_idx,
            )
        elif i == 750:
            # control first dof with velocity, and the rest with position
            franka.control_dofs_position(
                np.array([0, 0, 0, 0, 0, 0, 0, 0, 0])[1:],
                motors_dof_idx[1:],
            )
            franka.control_dofs_velocity(
                np.array([1.0, 0, 0, 0, 0, 0, 0, 0, 0])[:1],
                motors_dof_idx[:1],
            )
        elif i == 1000:
            franka.control_dofs_force(
                np.array([0, 0, 0, 0, 0, 0, 0, 0, 0]),
                motors_dof_idx,
            )
        scene.step()
        write_frame(n_hard_reset + i)
finally:
    ffmpeg.stdin.close()
    ffmpeg.wait()

n_frames = (n_hard_reset + horizon) // render_every
print(f"{png_path} | {mp4_path} at {fps} fps, {n_frames} frames")


[Genesis] [05:54:29] [INFO] Scene <4187060> created.
[Genesis] [05:54:29] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 0, uid: <b1c455a>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.


[Genesis] [05:54:29] [WARNING] (MJCF) Approximating tendon by joint actuator for `finger_joint1`
[Genesis] [05:54:29] [WARNING] (MJCF) Approximating tendon by joint actuator for `finger_joint2`
[Genesis] [05:54:29] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 1, uid: <59545ad>, morph: <gs.morphs.MJCF(file='/mnt/dcgpuval/younsroh/projects/3d/.venv/lib/python3.12/site-packages/genesis/assets/xml/franka_emika_panda/panda.xml')>, material: <gs.materials.Rigid>.
[Genesis] [05:54:29] [INFO] Building scene <4187060>...
[Genesis] [05:54:29] [WARNING] Neutral robot position (qpos0) exceeds joint limits.
[Genesis] [05:54:29] [WARNING] Constraint solver time constant should be greater than 2*substep_dt. timeconst is changed from `0.005` to `0.02`). Decrease simulation timestep or increase timeconst to avoid altering the original value.
[Genesis] [05:54:30] [WARNING] Filtered out geometry pairs causing self-collision for the neutral configuration (qpos0): (6, 11), (7, 11), (14, 21), (15, 2

### Pick and Place with Suction Cup

In [2]:
import subprocess
from pathlib import Path

import numpy as np
from PIL import Image

import genesis as gs

from genesis.ext.pyrender import constants as _pr_const, renderer as _pr_renderer, light as _pr_light

for _m in (_pr_const, _pr_renderer, _pr_light):
    _m.SHADOW_TEX_SZ = 2048

gs.init(backend=gs.amdgpu)

scene = gs.Scene(
    sim_options=gs.options.SimOptions(
        dt=0.01,
    ),
    rigid_options=gs.options.RigidOptions(
        box_box_detection=True,
    ),
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(0, -3.5, 2.5),
        camera_lookat=(0.0, 0.0, 0.5),
        camera_fov=30,
    ),
    show_viewer=False,
    vis_options=gs.options.VisOptions(
        shadow=True,
        ambient_light=(0.15, 0.15, 0.15),
        lights=[
            {"type": "directional", "dir": (-1, -1, -1), "color": (1.0, 1.0, 1.0), "intensity": 8.0},
            {"type": "point", "pos": (1.5, -1.0, 2.0), "color": (1.0, 0.95, 0.85), "intensity": 20.0},
        ],
    ),
)

plane = scene.add_entity(
    gs.morphs.Plane(),
)

cube = scene.add_entity(
    gs.morphs.Box(
        size=(0.04, 0.04, 0.04),
        pos=(0.65, 0.0, 0.02),
    ),
    surface=gs.surfaces.Plastic(
        color=(1, 0, 0),
    ),
)

cube_2 = scene.add_entity(
    gs.morphs.Box(
        size=(0.04, 0.04, 0.04),
        pos=(0.4, 0.2, 0.02),
    ),
    surface=gs.surfaces.Plastic(
        color=(0, 1, 0),
    ),
)

franka = scene.add_entity(
    gs.morphs.MJCF(
        file="xml/franka_emika_panda/panda.xml",
    ),
    vis_mode="collision",
)
cam = scene.add_camera(
    res=(640, 480),
    pos=(0, -3.5, 2.5),
    lookat=(0.0, 0.0, 0.5),
    fov=30,
)
scene.build()
scene.visualizer._rasterizer._renderer._is_software = False

motors_dof = np.arange(7)
fingers_dof = np.arange(7, 9)
end_effector = franka.get_link("hand")

# set control gains
franka.set_dofs_kp(
    np.array([4500, 4500, 3500, 3500, 2000, 2000, 2000, 100, 100]),
)
franka.set_dofs_kv(
    np.array([450, 450, 350, 350, 200, 200, 200, 10, 10]),
)
franka.set_dofs_force_range(
    np.array([-87, -87, -87, -87, -12, -12, -12, -100, -100]),
    np.array([87, 87, 87, 87, 12, 12, 12, 100, 100]),
)

out_dir = Path("out")
out_dir.mkdir(exist_ok=True)
png_path = out_dir / "suction_cup_t0.png"
mp4_path = out_dir / "suction_cup.mp4"

frame = cam.render()[0]
Image.fromarray(frame).save(png_path)

render_every = 4
height, width = frame.shape[:2]
fps = round(1.0 / (scene.dt * render_every))
step_idx = 0

ffmpeg = subprocess.Popen(
    [
        "ffmpeg", "-y", "-loglevel", "error",
        "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{width}x{height}", "-framerate", str(fps),
        "-i", "-",
        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18",
        str(mp4_path),
    ],
    stdin=subprocess.PIPE,
)


def write_frame():
    global step_idx
    if step_idx % render_every == 0:
        ffmpeg.stdin.write(np.ascontiguousarray(cam.render()[0]).tobytes())
    step_idx += 1

# move to pre-grasp pose
qpos = franka.inverse_kinematics(
    link=end_effector,
    pos=np.array([0.65, 0.0, 0.25]),
    quat=np.array([0, 1, 0, 0]),
)

# gripper open pos
qpos[-2:] = 0.04

path = franka.plan_path(
    qpos_goal=qpos,
    num_waypoints=100,  # 1s duration
)

try:
    # execute the planned path
    for waypoint in path:
        franka.control_dofs_position(waypoint)
        franka.control_dofs_force(np.array([0.5, 0.5]), fingers_dof)
        scene.step()
        write_frame()

    # allow robot to reach the last waypoint
    for i in range(100):
        scene.step()
        write_frame()

    # reach
    qpos = franka.inverse_kinematics(
        link=end_effector,
        pos=np.array([0.65, 0.0, 0.130]),
        quat=np.array([0, 1, 0, 0]),
    )
    franka.control_dofs_position(qpos[:-2], motors_dof)
    for i in range(50):
        scene.step()
        write_frame()

    # add suction / weld constraint
    rigid = scene.sim.rigid_solver
    link_cube = cube.get_link("box_baselink").idx
    link_franka = franka.get_link("hand").idx
    rigid.add_weld_constraint(link_cube, link_franka)

    # lift
    qpos = franka.inverse_kinematics(
        link=end_effector,
        pos=np.array([0.65, 0.0, 0.28]),
        quat=np.array([0, 1, 0, 0]),
    )
    franka.control_dofs_position(qpos[:-2], motors_dof)
    for i in range(50):
        scene.step()
        write_frame()

    # reach
    qpos = franka.inverse_kinematics(
        link=end_effector,
        pos=np.array([0.4, 0.2, 0.18]),
        quat=np.array([0, 1, 0, 0]),
    )
    franka.control_dofs_position(qpos[:-2], motors_dof)
    for i in range(100):
        scene.step()
        write_frame()

    # release
    rigid.delete_weld_constraint(link_cube, link_franka)
    for i in range(400):
        scene.step()
        write_frame()
finally:
    ffmpeg.stdin.close()
    ffmpeg.wait()

print(f"{png_path} | {mp4_path} at {fps} fps, {step_idx // render_every} frames")


[Genesis] [23:51:20] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [23:51:20] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [23:51:20] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [23:51:21] [INFO] Running on [AMD Instinct MI325X] with backend gs.amdgpu. Device memory: 255.98 GB.
[Genesis] [23:51:21] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.1, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
[Genesis] [23:51:21] [INFO] Scene <ae4da8d> created.
[Genesis] [23:51:21] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 0, uid: <33d171b>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.


/mnt/dcgpuval/younsroh/projects/3d/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[Genesis] [23:51:21] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 1, uid: <b69fb2e>, morph: <gs.morphs.Box>, material: <gs.materials.Rigid>.
[Genesis] [23:51:21] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 2, uid: <7137e8e>, morph: <gs.morphs.Box>, material: <gs.materials.Rigid>.
[Genesis] [23:51:22] [WARNING] (MJCF) Approximating tendon by joint actuator for `finger_joint1`
[Genesis] [23:51:22] [WARNING] (MJCF) Approximating tendon by joint actuator for `finger_joint2`
[Genesis] [23:51:22] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 3, uid: <e0b9b6d>, morph: <gs.morphs.MJCF(file='/mnt/dcgpuval/younsroh/projects/3d/.venv/lib/python3.12/site-packages/genesis/assets/xml/franka_emika_panda/panda.xml')>, material: <gs.materials.Rigid>.
[Genesis] [23:51:22] [INFO] Building scene <ae4da8d>...
[Genesis] [23:51:22] [WARNING] Neutral robot position (qpos0) exceeds joint limits.
[Genesis] [23:51:22] [WARNING] Constraint solver time constant should be greater than 2*subs

/mnt/dcgpuval/younsroh/projects/3d/.venv/lib/python3.12/site-packages/quadrants/_test_tools/warnings_helper.py:11: UserWarning: cannot create weak reference to 'tuple' object. Template mapper caching disabled.
  warnings.warn(message)


[Genesis] [23:51:45] [INFO] Running at 9.20 FPS.
[Genesis] [23:51:45] [INFO] Running at 16.18 FPS.
[Genesis] [23:51:45] [INFO] Running at 22.58 FPS.
[Genesis] [23:51:45] [INFO] Running at 27.70 FPS.
[Genesis] [23:51:45] [INFO] Running at 32.52 FPS.
[Genesis] [23:51:45] [INFO] Running at 37.27 FPS.
[Genesis] [23:51:45] [INFO] Running at 42.31 FPS.
[Genesis] [23:51:45] [INFO] Running at 47.20 FPS.
[Genesis] [23:51:45] [INFO] Running at 51.67 FPS.
[Genesis] [23:51:45] [INFO] Running at 55.74 FPS.
[Genesis] [23:51:46] [INFO] Running at 59.44 FPS.
[Genesis] [23:51:46] [INFO] Running at 63.36 FPS.
[Genesis] [23:51:46] [INFO] Running at 67.28 FPS.
[Genesis] [23:51:46] [INFO] Running at 71.29 FPS.
[Genesis] [23:51:46] [INFO] Running at 74.99 FPS.
[Genesis] [23:51:46] [INFO] Running at 78.35 FPS.
[Genesis] [23:51:46] [INFO] Running at 81.47 FPS.
[Genesis] [23:51:46] [INFO] Running at 84.01 FPS.
[Genesis] [23:51:46] [INFO] Running at 86.36 FPS.
[Genesis] [23:51:46] [INFO] Running at 88.61 FPS.
[

### Parallel Simulation

In [9]:
import subprocess
from pathlib import Path

import numpy as np
import torch
from PIL import Image

import genesis as gs

# Elevated viewpoint so the 5x4 env grid (env_spacing=1 m, centered) is in frame.
cam_pos = (4.0, -1.0, 2.0)  #(7.5, -7.5, 6.0)
cam_lookat = (1.0, 2.0, 0.4)  #(0.0, 0.0, 0.4)

scene = gs.Scene(
    sim_options=gs.options.SimOptions(
        dt=0.01,
    ),
    viewer_options=gs.options.ViewerOptions(
        camera_pos=cam_pos,
        camera_lookat=cam_lookat,
        camera_fov=40,
    ),
    show_viewer=False,
    vis_options=gs.options.VisOptions(
        # llvmpipe (CPU GL) on this node: leave shadows off so pyrender does not
        # allocate shadow maps. Physics still runs on gs.amdgpu.
        shadow=False,
        ambient_light=(0.15, 0.15, 0.15),
        lights=[
            {"type": "directional", "dir": (-1, -1, -1), "color": (1.0, 1.0, 1.0), "intensity": 8.0},
        ],
    ),
)

plane = scene.add_entity(
    gs.morphs.Plane(),
)

franka = scene.add_entity(
    gs.morphs.MJCF(file="xml/franka_emika_panda/panda.xml"),
)
cam = scene.add_camera(
    res=(640, 480),
    pos=cam_pos,
    lookat=cam_lookat,
    fov=40,
)

# create 20 parallel environments
B = 20
scene.build(n_envs=B, env_spacing=(1.0, 1.0))

# control all the robots
franka.control_dofs_position(
    torch.tile(torch.tensor([0, 0, 0, -1.0, 0, 1.0, 0, 0.02, 0.02], device=gs.device), (B, 1)),
)

# 'envs_idx' narrows a command to a subset of the environments, leaving the rest on their previous target
franka.control_dofs_position(
    torch.zeros(3, 9, device=gs.device),
    envs_idx=torch.tensor([1, 5, 7], device=gs.device),
)

out_dir = Path("out")
out_dir.mkdir(exist_ok=True)
png_path = out_dir / "parallel_sim_t0.png"
mp4_path = out_dir / "parallel_sim.mp4"

frame = cam.render()[0]
Image.fromarray(frame).save(png_path)

steps = 1000
render_every = 3
height, width = frame.shape[:2]
fps = round(1.0 / (scene.dt * render_every))

ffmpeg = subprocess.Popen(
    [
        "ffmpeg", "-y", "-loglevel", "error",
        "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{width}x{height}", "-framerate", str(fps),
        "-i", "-",
        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18",
        str(mp4_path),
    ],
    stdin=subprocess.PIPE,
)

try:
    for i in range(steps):
        scene.step()
        if i % render_every == 0:
            ffmpeg.stdin.write(np.ascontiguousarray(cam.render()[0]).tobytes())
finally:
    ffmpeg.stdin.close()
    ffmpeg.wait()

print(f"{png_path} | {mp4_path} at {fps} fps, {steps // render_every} frames")


[Genesis] [00:40:04] [INFO] Scene <e6b5427> created.
[Genesis] [00:40:04] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 0, uid: <b825fc8>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [00:40:04] [WARNING] (MJCF) Approximating tendon by joint actuator for `finger_joint1`
[Genesis] [00:40:04] [WARNING] (MJCF) Approximating tendon by joint actuator for `finger_joint2`
[Genesis] [00:40:04] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 1, uid: <9f6a2c6>, morph: <gs.morphs.MJCF(file='/mnt/dcgpuval/younsroh/projects/3d/.venv/lib/python3.12/site-packages/genesis/assets/xml/franka_emika_panda/panda.xml')>, material: <gs.materials.Rigid>.
[Genesis] [00:40:04] [INFO] Building scene <e6b5427>...
[Genesis] [00:40:04] [WARNING] Neutral robot position (qpos0) exceeds joint limits.
[Genesis] [00:40:04] [WARNING] Constraint solver time constant should be greater than 2*substep_dt. timeconst is changed from `0.005` to `0.02`). Decrease simulation timestep or increase

### Speed Benchmark - 30K Franka Environments

In [12]:
gs.destroy()

[Genesis] [01:50:16] [INFO] 💤 Exiting Genesis and caching compiled kernels...


In [ ]:
import torch

import genesis as gs

# The benchmark measures throughput at 30000 parallel environments, which needs a GPU.
gs.init(backend=gs.amdgpu, performance_mode=True)

scene = gs.Scene(
    sim_options=gs.options.SimOptions(
        dt=0.01,
    ),
    viewer_options=gs.options.ViewerOptions(
        res=(1920, 1080),
        camera_pos=(3.5, -1.0, 2.5),
        camera_lookat=(0.0, 0.0, 0.5),
        camera_fov=40,
    ),
    show_viewer=False,
)

plane = scene.add_entity(
    gs.morphs.Plane(),
)

franka = scene.add_entity(
    gs.morphs.MJCF(file="xml/franka_emika_panda/panda.xml"),
)


# create 30000 parallel environments
B = 30000
scene.build(n_envs=B, env_spacing=(1.0, 1.0))

# control all the robots
# with the following control: 43M FPS
# without the following control (arm in collision with the floor): 32M FPS
franka.control_dofs_position(
    torch.tile(torch.tensor([0, 0, 0, -1.0, 0, 1.0, 0, 0.02, 0.02], device=gs.device), (B, 1)),
)

for i in range(1000):
    scene.step()

# ... 8M FPS  on MI325X  --->  5X slower than 43M FPS on Nvidia box
# [Genesis] [01:51:10] [INFO] Running at 7,953,474.65 FPS (265.12 FPS per env, 30000 envs).


[Genesis] [01:50:23] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [01:50:23] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [01:50:23] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [01:50:23] [INFO] Running on [AMD Instinct MI325X] with backend gs.amdgpu. Device memory: 255.98 GB.
[Genesis] [01:50:23] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.1, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: True, 💬 verbose: INFO
[Genesis] [01:50:23] [INFO] Scene <024b9d1> created.
[Genesis] [01:50:23] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 0, uid: <418b070>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [01:50:23] [WARNING] (MJCF) Approximating tendon by joint actuator for `finger_joint1`
[Genesis] [01:50:23] [WARNING] (MJCF) Approximating tendon by joint actuator for `finger_joint2`
[Genesis] [01:50:24] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 1, uid: <7abb68

[W 09/25/26 01:50:26.836 2199666] QuadrantsWarning
While compiling `ext_arr_to_tensor_c14_11`, File "/mnt/dcgpuval/younsroh/projects/3d/.venv/lib/python3.12/site-packages/quadrants/_kernels.py", line 94, in ext_arr_to_tensor:
        tensor[I] = arr[I - offset]
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^
Assign may lose precision: i32 <- i64


[Genesis] [01:50:27] [WARNING] Constraint solver time constant should be greater than 2*substep_dt. timeconst is changed from `0.005` to `0.02`). Decrease simulation timestep or increase timeconst to avoid altering the original value.
[Genesis] [01:50:31] [WARNING] Filtered out geometry pairs causing self-collision for the neutral configuration (qpos0): (6, 11), (7, 11), (14, 21), (15, 20), (16, 23), (17, 22). Consider tuning Morph option 'decompose_robot_error_threshold' or specify dedicated collision meshes. This behavior can be disabled by setting Morph option 'enable_neutral_collision=True'.
[Genesis] [01:50:38] [INFO] Compiling simulation kernels...
[Genesis] [01:51:05] [INFO] Building visualizer...
[Genesis] [01:51:05] [INFO] Software rendering context detected. Shadows and plane reflection not supported. Beware rendering will be extremely slow.
[Genesis] [01:51:06] [INFO] Running at 6,849,677.71 FPS (228.32 FPS per env, 30000 envs).
[Genesis] [01:51:06] [INFO] Running at 6,850,6